# Using Predictive Analytics to Identify Engagement Drivers in Instagram Reels
## Data Understanding and Research
---
---

## Problem Statement (rewrite to specify this is for data research and understanding phase)
#### Audience engagement with short-form Instagram Reels varies substantially across videos, even within the same creator’s account. Because creators differ widely in audience size and baseline performance, comparisons based solely on raw engagement metrics can obscure the relationship between content-level characteristics and engagement outcomes.
#### The purpose of this analysis is to identify which content characteristics are most strongly associated with higher engagement in short-form Reels focused on DIY content, after controlling for creator-level consistency and baseline performance. Engagement is evaluated using both a creator-relative performance indicator (performance_bucket) and an engagement rate calculated as the ratio of likes and comments to views. This approach enables performance to be assessed relative to each creator’s typical output rather than across creators with unequal reach.
#### Predictive analytics can be applied to model the relationship between Reel-level features and the likelihood that a Reel achieves above-baseline engagement. Insights from this analysis can inform data-driven content strategies by identifying repeatable patterns associated with stronger audience interaction, independent of creator popularity.
---
---

## Data Dictionary

| Feature               | Description                                                              | Type of Variable            | Data Type        |
| --------------------- | ------------------------------------------------------------------------ | --------------------------- | ---------------- |
| reel_id               | Unique identifier assigned by Instagram                                  | Categorical                 | string           |
| creator_handle        | Unique account name                                                      | Categorical                 | string           |
| total_followers       | Total number of followers per account                                    | Numerical                   | int              |
| date_posted           | Date the Reel was published                                              | Numerical                   | datetime         |
| performance_bucket    | Indicates Reel performance as high or baseline (recent)                  | Categorical                 | string           |
| views                 | Total views per Reel                                                     | Numerical                   | int              |
| likes                 | Total likes per Reel                                                     | Numerical                   | int              |
| comments              | Total comments per Reel                                                  | Numerical                   | int              |
| engagement_rate       | Proportion of viewers who engaged via likes or comments                  | Numerical                   | float            |
| reel_structure        | List, process, transformation, narrative, or showcase                    | Categorical                 | string           |
| craft_category        | Type of craft presented in the Reel                                      | Categorical                 | string           |
| hook_category         | POV, Question, Declarative Statement, Transformation Tease, Relatable    | Categorical                 | string           |
| pov_present           | 1st (creator) or 2nd (place user in scenario) person POV framing         | Categorical                 | string           |
| results_shown_first   | Final result shown in the opening frames of the Reel during playback     | Categorical                 | string           |
| voice_present         | Spoken audio present                                                     | Categorical                 | string           |
| audio_not_required    | Message understandable without audio                                     | Categorical                 | string           |

## Data Imports

In [1]:
import pandas as pd
import numpy as np

In [2]:
reels_df = pd.read_csv('../Data/raw_data.csv')
reels_df.head(10)

,creator_handle,total_followers,url,date_posted,collection_date,performance_bucket,views_raw,likes_raw,comments_raw,engagement_rate,reel_structure,craft_category,hook_text,hook_category,pov_present,result_shown_first,voice_present,audio_not_required
0,annikamarieee,953K,https://www.instagram.com/reel/DT5we1ZAQS4/,1/24/2026,2/19/2026,baseline,113K,1199,13,NaN,Transformation,Home Decor,FAKE,transformation_tease,Y,Y,Y,Y
1,annikamarieee,953K,https://www.instagram.com/reel/DT3iSi1EyfE/,1/23/2026,2/19/2026,baseline,36.5K,1034,8,NaN,Transformation,Home Decor,how it started vs. how it's going,transformation_tease,Y,N,N,Y
2,annikamarieee,953K,https://www.instagram.com/reel/DTgI-ZoEpwJ/,1/14/2026,2/19/2026,baseline,162K,4619,44,NaN,Process,Home Decor,Can't look at this floor one more day...,transformation_tease,N,N,N,N
3,annikamarieee,953K,https://www.instagram.com/reel/DSVtT-_k5Vm/,12/16/2025,2/19/2026,baseline,285K,8650,54,NaN,Process,Home Decor,NaN,transformation_tease,N,Y,Y,Y
4,annikamarieee,953K,https://www.instagram.com/reel/DSIemD0D07c/,12/11/2025,2/19/2026,baseline,554K,6546,25,NaN,Process,Home Decor,I'm going on a year now without having a dinin...,transformation_tease,N,N,Y,Y
5,annikamarieee,953K,https://www.instagram.com/reel/DMijUhrJqBJ/,7/25/2025,2/19/2026,baseline,615K,28.1K,130,NaN,Process,Home Decor,So I have been really wanting to try out stain...,curiosity_gap,N,N,Y,Y
6,annikamarieee,953K,https://www.instagram.com/reel/DKcmKpDpFLk/,6/3/2025,2/19/2026,baseline,173K,4271,21,NaN,Process,Fine Art / Paint,Comment: We need a Girly Pop Workshop sign,relatable_observation,N,N,N,N
7,annikamarieee,953K,https://www.instagram.com/reel/DJmfm0Xpvcg/,5/13/2025,2/19/2026,high,6.1M,322K,365,NaN,Process,Home Decor,Just throw it away. Me: But it has potential!,transformation_tease,N,N,N,Y
8,annikamarieee,953K,https://www.instagram.com/reel/DJHq-AnvZ7n/,5/1/2025,2/19/2026,high,6.1M,410K,1589,NaN,Process,Plants / Garden,I've been dreaming of a little potting station...,direct_value_delivery,N,Y,Y,Y
9,annikamarieee,953K,https://www.instagram.com/reel/DJUeF9rPTIQ/,5/6/2025,2/19/2026,high,2.2M,185K,530,NaN,Transformation,Home Decor,POV: You made custom shelves for your cat. She...,direct_value_delivery,N,N,N,Y


---

## Data Cleaning
#### Data cleaning will focus on ensuring consistency, accuracy, and interpretability of Reel-level content features prior to analysis.
* Unique Reel identifiers will be extracted from the URL to create a unique identifier and prevent duplicate entries. Date fields will be standardized to a consistent format to support basic temporal interpretation.
* Missing values will be examined across all variables. In cases where hook_text is not present because the hook is conveyed visually, values will be left missing rather than imputed. This variable will be retained for documentation purposes but excluded from predictive modeling.
* Binary delivery-mechanic variables will be reviewed to ensure consistent categorical encoding prior to modeling.
* The following will be converted from strings to numerical values:
    * total_followers
    * views
    * likes
    * comments
* An engagement_rate variable will be calculated using likes and comments relative to views to allow engagement comparisons across creators with different audience sizes.
* Categorical variables including hook_category, reel_structure, and craft_category will be reviewed and refined to ensure categories are mutually exclusive and consistently applied.
---

## Feature Selection
##### The objective of this analysis is to identify which Reel-level content characteristics are most strongly associated with higher engagement in short-form DIY Reels, while controlling for creator-level consistency and baseline performance. Features were selected to capture how content is framed, structured, and delivered, rather than relying solely on aggregate popularity metrics.

### Target Variable
##### The primary target variable is **performance_bucket**, a categorical variable indicating whether a Reel is classified as high-performing or baseline relative to the creator’s typical performance. Defining success within each creator’s account reduces bias introduced by differences in follower count and reach, allowing engagement to be evaluated in a more comparable context. In addition, **engagement_rate**, calculated as (likes + comments) / views, is retained as a supporting quantitative metric to contextualize engagement intensity and validate differences between performance categories.

### Predictor Variables
##### Predictor variables were selected based on their ability to vary across Reels within the same creator and their relevance to content framing and delivery decisions.

#### Content Framing and Structure
* **hook_category:** Categorizes the semantic framing used to attract attention at the start of the Reel (POV / Role Framing, Question / Curiosity Gap, Declarative Statement, Transformation Tease, Pain Point / Relatable).
* **reel_structure:** Describes how the content is organized (list, process, transformation, narrative, showcase).
* **craft_category:** Identifies the type of DIY content presented.

##### These features capture how creators contextualize and organize content, which may influence viewer interest and engagement.

#### Delivery Mechanics
* **results_shown_first:** Indicates whether the final outcome is visible in the opening frames of the Reel.
* **voice_present:** Indicates whether spoken audio is present.
* **audio_not_required:** Indicates whether the Reel can be fully understood without audio.
* **pov_present:** Indicates whether the viewer is framed in a first- or second-person perspective.

##### These variables capture presentation choices that affect accessibility, clarity, and immediate viewer comprehension.

### Control Variables
* **total_followers** is included to account for differences in creator audience size.
* **date_posted** is retained to preserve temporal context and allow for potential analysis related to recency effects.

### Excluded Variables
* **reel_id** and **creator_handle** are retained for identification and grouping but are not used as predictors.
* **views**, **likes**, and **comments** are excluded as predictors because they are components of the engagement outcome.
* **hook_text** is retained for documentation purposes but excluded from modeling due to missing values and subjective variation.

### Variable Type Clarification
##### Although **performance_bucket** and several delivery-mechanic variables contain only two possible values, they are treated as categorical variables rather than numerical. These variables represent class membership or the presence or absence of a condition, and no ordinal or quantitative relationship is assumed between categories.